# ECON 662 — Problem Set 2
## Part (c.2): Forward Simulation MLE Using Equations (6) and (7)

Estimate $(\theta,\rho,\sigma_\rho)$ using the **second forward-simulation approach** in the homework notes.

The problem set says:

- the first two steps are the same as Part (b),
- for Part (c.2), use Equation (6) and Equation (7),
- then compute continuation values, form model CCPs, and maximize the likelihood.

We also follow the methodological structure from Lecture 15:

1. Estimate transitions.
2. Estimate CCPs from the data.
3. For each guess of $\theta$, approximate the value function by forward simulation.
4. Use the approximated value function to form model CCPs and then compute the MLE objective.


### Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.random.seed(42)


### Load Data


In [2]:
# Variables:
# i  = bus id
# t  = month
# a  = action (0 = maintain, 1 = replace)
# x  = mileage state in {1, ..., 7}
# rc = replacement cost (continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)


Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


### Global Constants


In [3]:
BETA      = 0.95
N_X       = 7
N_RC_BINS = 8
S         = N_X * N_RC_BINS

# Forward-simulation tuning choices for Part (c.2)
N_SIM     = 100
T_SIM     = 120
SIM_SEED  = 24680

print(f"beta        = {BETA}")
print(f"N_X         = {N_X}")
print(f"N_RC_BINS   = {N_RC_BINS}")
print(f"|S|         = {S}")
print(f"N_SIM       = {N_SIM}")
print(f"T_SIM       = {T_SIM}")
print(f"beta^T_SIM  = {BETA**T_SIM:.6f}")


beta        = 0.95
N_X         = 7
N_RC_BINS   = 8
|S|         = 56
N_SIM       = 100
T_SIM       = 120
beta^T_SIM  = 0.002122


### Estimate the RC Process and Build the State Grid

As in Parts (b) and (c.1), we keep the first-step objects fixed throughout the MLE:

1. Build an empirical RC grid with equal-width bins.
2. Estimate the RC transition matrix $\Pi$ directly from observed bin-to-bin movements.
3. Estimate the continuous AR(1) process

$$
RC_t = \rho_0 + \rho_1 RC_{t-1} + e_t,\qquad e_t \sim N(0,\sigma_\rho^2),
$$
by OLS.


In [4]:
# Equal-width RC bins from the observed support
rc_min = df['rc'].min()
rc_max = df['rc'].max()
bin_edges = np.linspace(rc_min, rc_max, N_RC_BINS + 1)
rc_grid = (bin_edges[:-1] + bin_edges[1:]) / 2.0

# Assign each observation to a bin (0-indexed)
rc_bin_idx = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)
df['rc_bin'] = rc_bin_idx
df['rc_mid'] = rc_grid[rc_bin_idx]

print("RC bin edges:")
print(np.round(bin_edges, 3))
print("\nRC grid (bin midpoints):")
print(np.round(rc_grid, 3))

# Empirical RC transition matrix Pi
C = np.zeros((N_RC_BINS, N_RC_BINS))
for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    for j_from, j_to in zip(bins[:-1], bins[1:]):
        C[j_from, j_to] += 1

Pi = C / C.sum(axis=1, keepdims=True)

print("\nEmpirical RC transition matrix Pi:")
print(np.round(Pi, 4))
print("Row sums:")
print(np.round(Pi.sum(axis=1), 10))

# OLS for the continuous AR(1) process
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    vals = g['rc'].values
    rc_prev_list.append(vals[:-1])
    rc_curr_list.append(vals[1:])

rc_prev = np.concatenate(rc_prev_list)
rc_curr = np.concatenate(rc_curr_list)

X_ols = np.column_stack([np.ones(len(rc_prev)), rc_prev])
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ rc_curr)
rho0_hat, rho1_hat = beta_ols
sigma_rho_hat = (rc_curr - X_ols @ beta_ols).std(ddof=1)

print("\nAR(1) estimates from continuous RC:")
print(f"rho0_hat      = {rho0_hat:.4f}")
print(f"rho1_hat      = {rho1_hat:.4f}")
print(f"sigma_rho_hat = {sigma_rho_hat:.4f}")


RC bin edges:
[31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]

RC grid (bin midpoints):
[33.438 37.312 41.188 45.062 48.938 52.812 56.688 60.562]

Empirical RC transition matrix Pi:
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.     0.     0.     0.     0.2857 0.5714 0.     0.1429]
 [0.     0.     0.     0.     0.     0.5    0.5    0.    ]]
Row sums:
[1. 1. 1. 1. 1. 1. 1. 1.]

AR(1) estimates from continuous RC:
rho0_hat      = 17.8269
rho1_hat      = 0.6249
sigma_rho_hat = 4.6060


### Build the Full Transition Matrices and Estimate CCPs

The combined state is $s=(x,j)$, where:

- $x \in \{1,\dots,7\}$ is mileage,
- $j \in \{0,\dots,7\}$ is the RC bin.

Hence the model has

$$
|S| = 7 \times 8 = 56
$$

discrete states.

The state transitions are:

- if $a=0$ (maintain), then $x'=\min(x+1,7)$,
- if $a=1$ (replace), then $x'=1$,
- and the RC-bin transition is given by the empirical matrix $\Pi$.

We also estimate the conditional choice probabilities using state frequencies:

$$
\hat P(a=1\mid s)=\frac{N_1(s)}{N_0(s)+N_1(s)},
\qquad
\hat P(a=0\mid s)=1-\hat P(a=1\mid s).
$$

As in Part (c.1), we clip the estimated CCPs slightly away from 0 and 1 so the log-odds are always finite.


In [5]:
# Flat state indexing: s = xi * N_RC_BINS + j
xi_of_s = np.array([xi for xi in range(N_X) for _ in range(N_RC_BINS)])
j_of_s  = np.array([j  for _ in range(N_X) for j in range(N_RC_BINS)])
x_of_s  = xi_of_s + 1
rc_of_s = rc_grid[j_of_s]

# Next mileage under each action
xi_next_mnt = np.minimum(xi_of_s + 1, N_X - 1)
xi_next_rep = np.zeros(S, dtype=int)

# Action-specific transition matrices TP0 and TP1
TP0 = np.zeros((S, S))
TP1 = np.zeros((S, S))
for s in range(S):
    j = j_of_s[s]

    col0 = xi_next_mnt[s] * N_RC_BINS
    TP0[s, col0:col0 + N_RC_BINS] = Pi[j, :]

    col1 = xi_next_rep[s] * N_RC_BINS
    TP1[s, col1:col1 + N_RC_BINS] = Pi[j, :]

print(f"TP0 shape: {TP0.shape}")
print(f"TP1 shape: {TP1.shape}")
print(f"TP0 row sums: [{TP0.sum(1).min():.8f}, {TP0.sum(1).max():.8f}]")
print(f"TP1 row sums: [{TP1.sum(1).min():.8f}, {TP1.sum(1).max():.8f}]")

# State-level counts
N1 = np.zeros((N_X, N_RC_BINS))
N0 = np.zeros((N_X, N_RC_BINS))

for xi_obs, j_obs, a_obs in zip(df['x'].values.astype(int) - 1,
                                df['rc_bin'].values.astype(int),
                                df['a'].values.astype(int)):
    if a_obs == 1:
        N1[xi_obs, j_obs] += 1
    else:
        N0[xi_obs, j_obs] += 1

N_total = N0 + N1

eps_ccp = 1e-6
P1_hat = np.where(N_total > 0, N1 / N_total, eps_ccp)
P1_hat = np.clip(P1_hat, eps_ccp, 1.0 - eps_ccp)
P0_hat = 1.0 - P1_hat

P1_flat = P1_hat.ravel()
P0_flat = P0_hat.ravel()
log_odds_flat = np.log(P1_flat) - np.log(P0_flat)

print("\nEstimated CCP  P_hat(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")

print(f"\nTotal observations recovered from N0+N1: {N_total.sum():.0f}")


TP0 shape: (56, 56)
TP1 shape: (56, 56)
TP0 row sums: [1.00000000, 1.00000000]
TP1 row sums: [1.00000000, 1.00000000]

Estimated CCP  P_hat(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2: [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3: [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4: [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5: [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6: [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7: [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]

Total observations recovered from N0+N1: 100000


### Proof of Equation (6)

Let the deterministic choice-specific value be

$$
\delta_0(s;\theta)=\bar u(s,0;\theta)+\beta EV(s,0;\theta),
\qquad
\delta_1(s;\theta)=\bar u(s,1;\theta)+\beta EV(s,1;\theta).
$$

The agent chooses action 1 if and only if

$$
\delta_1(s;\theta)+\varepsilon_1 > \delta_0(s;\theta)+\varepsilon_0.
$$

Rearranging:

$$
\delta_1(s;\theta)-\delta_0(s;\theta)+\varepsilon_1-\varepsilon_0 > 0.
$$

Under Type I EV shocks, the corresponding logit CCP is

$$
P(a=1\mid s)=
\frac{\exp(\delta_1(s;\theta))}
{\exp(\delta_0(s;\theta))+\exp(\delta_1(s;\theta))},
\qquad
P(a=0\mid s)=
\frac{\exp(\delta_0(s;\theta))}
{\exp(\delta_0(s;\theta))+\exp(\delta_1(s;\theta))}.
$$

Therefore

$$
\frac{P(a=1\mid s)}{P(a=0\mid s)}
=
\exp\bigl(\delta_1(s;\theta)-\delta_0(s;\theta)\bigr),
$$

so

$$
\delta_1(s;\theta)-\delta_0(s;\theta)
=
\ln\!\left(\frac{P(a=1\mid s)}{P(a=0\mid s)}\right).
$$

Substituting this back into the choice rule gives

$$
\text{action}=1
\iff
\ln\!\left(\frac{P(a=1\mid s)}{P(a=0\mid s)}\right)
+ \varepsilon_1-\varepsilon_0 > 0,
$$

which is exactly Equation (6). In the simulation code we replace the true CCP by the first-step estimate $\hat P$, giving

$$
\text{action}=1
\iff
\ln\!\left(\frac{\hat P(a=1\mid s)}{\hat P(a=0\mid s)}\right)
+ \varepsilon_1-\varepsilon_0 > 0.
$$

Because the event of exact equality has probability zero under continuous shocks, using $>$ versus $\geq$ is trivial.


### Forward Simulation Using Direct Shock Draws

Now we implement the Part (c.2) simulation step exactly as in Equations (6) and (7).

At each simulated period and current state $s_\tau$:

1. Draw
   $$
   \varepsilon_{\tau,0}, \varepsilon_{\tau,1}
   \sim \text{Type I Extreme Value}
   $$
   independently.
2. Choose the action using Equation (6):
   $$
   a_\tau = 1
   \iff
   \ln\!\left(\frac{\hat P(a=1\mid s_\tau)}{\hat P(a=0\mid s_\tau)}\right)
   + \varepsilon_{\tau,1} - \varepsilon_{\tau,0} > 0.
   $$
3. Transition to the next state using the estimated transition matrix conditional on the chosen action.
4. Add the simulated payoff from Equation (7):
   $$
   \sum_{\tau=0}^{T_{\text{sim}}-1}
   \beta^\tau
   \left[
   \bar u(s_\tau,a_\tau;\theta) + \varepsilon_{\tau,a_\tau}
   \right].
   $$

Once again, because the estimated CCPs are fixed before the optimizer runs, the whole simulated path is independent of $\theta$ once the shocks are drawn. That means we can pre-compute:

$$
\hat V(s;\theta)
=
-\theta_1 M_s
-\theta_2 R_s
+ E_s,
$$

where

- $M_s$ is the simulated discounted mileage cost component,
- $R_s$ is the simulated discounted replacement-cost component,
- $E_s$ is the simulated discounted selected-shock component $\varepsilon_{\tau,a_\tau}$.


In [6]:
discounts = BETA ** np.arange(T_SIM)
cumTP0 = np.cumsum(TP0, axis=1)
cumTP1 = np.cumsum(TP1, axis=1)
cumTP0[:, -1] = 1.0
cumTP1[:, -1] = 1.0


def simulate_value_components_eq67():
    # Pre-compute the simulation objects in Equation (7) using common random numbers.
    rng = np.random.default_rng(SIM_SEED)

    maint_component = np.zeros(S)
    repl_component = np.zeros(S)
    shock_component = np.zeros(S)

    for s0 in range(S):
        states = np.full(N_SIM, s0, dtype=int)

        maint_acc = np.zeros(N_SIM)
        repl_acc = np.zeros(N_SIM)
        shock_acc = np.zeros(N_SIM)

        for tau in range(T_SIM):
            beta_tau = discounts[tau]

            eps0 = rng.gumbel(loc=0.0, scale=1.0, size=N_SIM)
            eps1 = rng.gumbel(loc=0.0, scale=1.0, size=N_SIM)

            log_odds = log_odds_flat[states]
            a1 = (log_odds + eps1 - eps0) > 0.0
            a0 = ~a1

            maint_acc += beta_tau * x_of_s[states] * a0
            repl_acc += beta_tau * rc_of_s[states] * a1

            selected_eps = np.where(a1, eps1, eps0)
            shock_acc += beta_tau * selected_eps

            u_next = rng.random(N_SIM)
            cdf = np.where(a1[:, None], cumTP1[states, :], cumTP0[states, :])
            next_states = (u_next[:, None] > cdf).sum(axis=1)
            next_states = np.minimum(next_states, S - 1)
            states = next_states

        maint_component[s0] = maint_acc.mean()
        repl_component[s0] = repl_acc.mean()
        shock_component[s0] = shock_acc.mean()

    return maint_component, repl_component, shock_component


t0_sim = time.time()
maint_component, repl_component, shock_component = simulate_value_components_eq67()
t_simulation = time.time() - t0_sim

print(f"Forward simulation pre-computation finished in {t_simulation:.2f} seconds.")
print(f"maint_component shape: {maint_component.shape}")
print(f"repl_component  shape: {repl_component.shape}")
print(f"shock_component shape: {shock_component.shape}")

print("\nFirst 10 entries of simulated components:")
print("maint_component:", np.round(maint_component[:10], 3))
print("repl_component :", np.round(repl_component[:10], 3))
print("shock_component:", np.round(shock_component[:10], 3))


Forward simulation pre-computation finished in 0.22 seconds.
maint_component shape: (56,)
repl_component  shape: (56,)
shock_component shape: (56,)

First 10 entries of simulated components:
maint_component: [51.079 51.43  51.076 52.839 52.833 51.478 51.713 52.395 51.552 51.604]
repl_component : [142.25  140.857 142.346 138.855 139.003 141.891 142.755 139.037 151.412
 150.338]
shock_component: [17.921 17.905 17.314 17.682 18.265 17.944 17.211 16.924 18.303 19.401]


### From Simulated $\hat V(s;\theta)$ to Model CCPs and the Likelihood

After simulating the value function, we compute action-specific continuation values:

$$
EV(s,0;\theta)=\sum_{s'} TP_0(s,s')\hat V(s';\theta),
\qquad
EV(s,1;\theta)=\sum_{s'} TP_1(s,s')\hat V(s';\theta).
$$

The deterministic choice-specific values are then

$$
\delta_0(s;\theta)=-\theta_1 x_s + \beta EV(s,0;\theta),
\qquad
\delta_1(s;\theta)=-\theta_2 RC_s + \beta EV(s,1;\theta).
$$

Under the logit formula, the model-implied CCP is

$$
\tilde P(a=1\mid s;\theta)
=
\frac{\exp(\delta_1(s;\theta))}
{\exp(\delta_0(s;\theta))+\exp(\delta_1(s;\theta))}.
$$

Finally we maximize the same state-level MLE objective as in Parts (a), (b), and (c.1):

$$
\ell(\theta_1,\theta_2)
=
\sum_{s=1}^{|S|}
\left[
N_1(s)\ln \tilde P(a=1\mid s;\theta)
+
N_0(s)\ln \tilde P(a=0\mid s;\theta)
\right].
$$


In [7]:
def simulated_value(theta1, theta2):
    # Equation (7) approximation:
    # V_hat(s;theta) = -theta1 * maint_component
    #                  -theta2 * repl_component
    #                  +shock_component
    return -theta1 * maint_component - theta2 * repl_component + shock_component


def model_ccp(theta1, theta2):
    V_hat = simulated_value(theta1, theta2)

    EV0 = TP0 @ V_hat
    EV1 = TP1 @ V_hat

    delta0 = -theta1 * x_of_s + BETA * EV0
    delta1 = -theta2 * rc_of_s + BETA * EV1

    dmax = np.maximum(delta0, delta1)
    exp0 = np.exp(delta0 - dmax)
    exp1 = np.exp(delta1 - dmax)
    P1_model = exp1 / (exp0 + exp1)

    return V_hat, P1_model.reshape(N_X, N_RC_BINS)


def neg_log_likelihood(params):
    theta1, theta2 = params

    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    _, P1_model = model_ccp(theta1, theta2)
    P0_model = 1.0 - P1_model

    eps = 1e-12
    ll = np.sum(N1 * np.log(P1_model + eps) + N0 * np.log(P0_model + eps))
    return -ll


theta_init = np.array([1.0, 1.0])
V_init, P1_init = model_ccp(theta_init[0], theta_init[1])

print("Initial-guess diagnostics at theta1=1, theta2=1")
print(f"NLL(theta_init) = {neg_log_likelihood(theta_init):.4f}")
print("\nFirst 10 entries of V_hat(theta_init):")
print(np.round(V_init[:10], 3))

print("\nModel-implied P(a=1|x,rc_bin) at theta1=theta2=1:")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_init[xi], 3)}")


Initial-guess diagnostics at theta1=1, theta2=1
NLL(theta_init) = 197218.8643

First 10 entries of V_hat(theta_init):
[-175.408 -174.382 -176.108 -174.012 -173.572 -175.425 -177.257 -174.509
 -184.662 -182.54 ]

Model-implied P(a=1|x,rc_bin) at theta1=theta2=1:
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3: [0.001 0.    0.    0.    0.    0.    0.    0.   ]
  x=4: [0.004 0.    0.    0.    0.    0.    0.    0.   ]
  x=5: [0.302 0.002 0.001 0.    0.    0.    0.    0.   ]
  x=6: [0.27  0.009 0.002 0.    0.    0.    0.    0.   ]
  x=7: [0.502 0.024 0.004 0.001 0.    0.    0.    0.   ]


### Optimization

For comparability with the earlier parts, we again use Nelder-Mead.

The optimizer now treats the first-step objects, the simulated shock draws, and the simulated paths as fixed. Each optimizer call only:

1. forms $\hat V(s;\theta)$,
2. computes continuation values and model CCPs,
3. evaluates the state-level log-likelihood.


In [8]:
print("--- Part (c.2): Forward Simulation with Direct Shock Draws ---")
print(f"Initial theta = {theta_init}")
print(f"Initial NLL   = {neg_log_likelihood(theta_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    theta_init,
    method="Nelder-Mead",
    options=dict(maxiter=5000, xatol=1e-8, fatol=1e-8, disp=True),
)
optimizer_runtime_sec = time.perf_counter() - t0_theta
optimizer_runtime_ms = 1000.0 * optimizer_runtime_sec
running_time_sec = t_simulation + optimizer_runtime_sec
running_time_ms = 1000.0 * running_time_sec

print(f"\nOptimizer-only time: {optimizer_runtime_sec:.4f} seconds ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")


--- Part (c.2): Forward Simulation with Direct Shock Draws ---
Initial theta = [1. 1.]
Initial NLL   = 197218.8643

Optimization terminated successfully.
         Current function value: 34408.221935
         Iterations: 81
         Function evaluations: 159

Optimizer-only time: 0.0039 seconds (3.93 ms)
Comparable running time: 0.2248 seconds (224.76 ms)


### Results


In [9]:
theta1_hat, theta2_hat = result.x
V_hat_final, P1_final = model_ccp(theta1_hat, theta2_hat)

print("=" * 64)
print("     PART (c.2) FORWARD SIMULATION -- DIRECT SHOCK DRAWS")
print("=" * 64)
print(f"theta1_hat    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"theta2_hat    (replacement cost coeff.) = {theta2_hat:.4f}")
print("-" * 64)
print("Pre-estimated RC process parameters:")
print(f"rho0_hat                                  = {rho0_hat:.4f}")
print(f"rho1_hat                                  = {rho1_hat:.4f}")
print(f"sigma_rho_hat                             = {sigma_rho_hat:.4f}")
print("-" * 64)
print(f"Choice log-likelihood                     = {-result.fun:.4f}")
print(f"Converged                                 = {result.success}")
print(f"Optimizer iterations                      = {result.nit}")
print(f"Forward-simulation precomputation time    = {t_simulation:.4f} sec")
print(f"Optimizer-only time                       = {optimizer_runtime_sec:.4f} sec ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time                   = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 64)

print("\nEstimated model CCP  P(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_final[xi], 3)}")


     PART (c.2) FORWARD SIMULATION -- DIRECT SHOCK DRAWS
theta1_hat    (mileage cost coeff.)     = 0.3897
theta2_hat    (replacement cost coeff.) = 0.1537
----------------------------------------------------------------
Pre-estimated RC process parameters:
rho0_hat                                  = 17.8269
rho1_hat                                  = 0.6249
sigma_rho_hat                             = 4.6060
----------------------------------------------------------------
Choice log-likelihood                     = -34408.2219
Converged                                 = True
Optimizer iterations                      = 81
Forward-simulation precomputation time    = 0.2208 sec
Optimizer-only time                       = 0.0039 sec (3.93 ms)
Comparable running time                   = 0.2248 sec (224.76 ms)

Estimated model CCP  P(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.02  0.01  0.01  0.006 0.003 0.002 0.001 0.   ]
  x=2: [0.109 0.054 0.058 

### Summary

The first two steps are the same as in Parts (b) and (c.1):

1. Estimate the RC transition process and discretize RC with empirical bins.
2. Estimate the CCP from the data.

The difference is the simulation step:

- Part (c.1): draw actions from $\hat P(a\mid s)$ and add the correction term $\gamma - \ln \hat P(a\mid s)$.
- Part (c.2): draw the Type I EV shocks directly, use Equation (6) to choose the action, and add the realized selected shock $\varepsilon_a$ as in Equation (7).
